# Baseline ResNet-50 — TCC ViT para Alzheimer (OASIS-1)

Este notebook executa o treino do **baseline ResNet-50 com pesos ImageNet** conforme as decisões registradas em [`docs/decisions/`](https://github.com/ThiagoCB1900/TCC/tree/main/docs/decisions). O ambiente alvo é o **Google Colab gratuito** (GPU T4).

## Pré-requisitos (faça uma vez antes da primeira execução)

1. **Subir a pasta `Data/` para o seu Google Drive** em `MyDrive/TCC/Data/` (1,3 GB, ~10-15 min de upload via Drive web).
    - Estrutura esperada: `MyDrive/TCC/Data/Non Demented/OAS1_*.jpg`, etc.
2. **Verificar que o repositório está atualizado em GitHub:** https://github.com/ThiagoCB1900/TCC (público).
3. **Garantir runtime com GPU:** `Runtime → Change runtime type → GPU (T4)`.

## O que este notebook faz

1. Verifica GPU e instala dependências (PyTorch já vem no Colab; só completamos com timm, MONAI, etc.).
2. Monta o Google Drive (acesso aos dados + persistência dos outputs).
3. Clona o repo do GitHub.
4. Liga `Data/` (via symlink ao Drive) e `experiments/runs/` (idem) — assim os outputs salvam direto no Drive e não se perdem se a sessão cair.
5. **Smoke test no GPU** (~2 min) para validar setup antes do treino real.
6. **Treino completo** (~1,5-3h em T4 com early stopping).
7. Análise dos resultados: curvas de loss/balanced accuracy, matriz de confusão.
8. Ablação obrigatória sem pesos de classe (ADR-0007).
9. Próximos passos.

## 1. Verificar GPU

Esperado: T4 com ~15 GB de VRAM (Colab gratuito). Se não aparecer GPU, vá em `Runtime → Change runtime type → GPU`.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponivel: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Montar Google Drive

Você vai precisar autorizar o Colab no popup. Após autorizar, deve aparecer `Mounted at /content/drive`.

Se a pasta `MyDrive/TCC/Data/` não existir, **pare aqui, faça o upload, e retome**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/TCC'
DRIVE_DATA = f'{DRIVE_ROOT}/Data'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'

assert os.path.isdir(DRIVE_DATA), (
    f'Pasta {DRIVE_DATA} nao encontrada no Drive. '
    'Faca upload do Data/ para MyDrive/TCC/ antes de continuar.'
)
os.makedirs(DRIVE_RUNS, exist_ok=True)

# Conta JPGs por classe para confirmar que e o dataset correto.
for cls in ['Non Demented', 'Very mild Dementia', 'Mild Dementia', 'Moderate Dementia']:
    path = f'{DRIVE_DATA}/{cls}'
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f'  {cls:25s} {n:>6d} arquivos')
print("\nEsperado total ~86.437 slices.")

## 3. Clonar repositório

Repo público em https://github.com/ThiagoCB1900/TCC . Se já estiver clonado, fazemos `pull`.

In [ ]:
%cd /content
if not os.path.isdir('/content/TCC'):
    !git clone https://github.com/ThiagoCB1900/TCC.git
else:
    %cd /content/TCC
    !git fetch --all && git reset --hard origin/main
%cd /content/TCC
!git log --oneline -5

## 4. Ligar dados e outputs ao Drive

- `Data/` no repo → symlink para `MyDrive/TCC/Data/`.
- `experiments/runs/` no repo → symlink para `MyDrive/TCC/runs/`. Assim cada checkpoint, history e config salva direto no Drive (persiste entre sessões).

In [ ]:
# Symlink Data/
!rm -rf /content/TCC/Data
!ln -s '/content/drive/MyDrive/TCC/Data' /content/TCC/Data
!ls /content/TCC/Data | head -5

# Symlink experiments/runs/
!mkdir -p /content/TCC/experiments
!rm -rf /content/TCC/experiments/runs
!ln -s '/content/drive/MyDrive/TCC/runs' /content/TCC/experiments/runs
!ls /content/TCC/experiments/

## 5. Instalar dependências

PyTorch já vem no Colab (versão com CUDA). Vamos só instalar timm, MONAI, etc. — sem reinstalar PyTorch.

In [ ]:
# Instala somente as libs do nosso projeto (timm, monai, etc.), sem mexer no PyTorch do Colab
!pip install -q timm==1.0.11 monai==1.3.2 torchmetrics==1.4.3 pypdf==5.0.0 tabulate==0.9.0 seaborn==0.13.2

In [ ]:
# Smoke import
import torch, timm, monai, torchmetrics
print('torch       :', torch.__version__, 'CUDA' if torch.cuda.is_available() else 'CPU')
print('timm        :', timm.__version__)
print('monai       :', monai.__version__)
print('torchmetrics:', torchmetrics.__version__)

## 6. Verificar estrutura

Antes de gastar GPU, confirmo que manifest e split existem e que o `OASISDataset` carrega corretamente sobre o Drive.

In [ ]:
%cd /content/TCC
import sys
if '/content/TCC' not in sys.path:
    sys.path.insert(0, '/content/TCC')

assert os.path.isfile('results/eda/manifest.csv'), 'manifest.csv ausente — repo desatualizado?'
assert os.path.isfile('experiments/splits/split_v1.json'), 'split_v1.json ausente'

from src.data.dataset import OASISDataset
train_ds = OASISDataset(
    manifest_path='results/eda/manifest.csv',
    split_path='experiments/splits/split_v1.json',
    fold='train',
    label_scheme='class_3',
    data_root='.',
)
print(f'train fold: {len(train_ds)} slices, {train_ds.subject_counts()} pacientes')
print(f'class_to_idx: {train_ds.class_to_idx}')
print(f'class_counts: {train_ds.class_counts()}')

# Espera-se 60.329 slices, 242 pacientes, class_to_idx {non=0, very_mild=1, mild_or_moderate=2}

## 7. Smoke test no GPU

Validação rápida (~1-2 min) do pipeline com GPU antes do treino real. Esperado:
- `train_loss` e `val_loss` diminuem entre as duas épocas
- matriz de confusão tem amostras das 3 classes
- nenhum erro de OOM / CUDA

In [ ]:
!python -m src.training.run \
    --smoke --smoke-batches 30 \
    --batch-size 32 --batch-size-eval 64 \
    --num-workers 2 \
    --run-name resnet50_smoke_colab

## 8. Treino completo do baseline (weighted CE)

Configuração principal conforme ADR-0007: **com** pesos de classe `[0.43, 2.11, 4.92]`. Early stopping em `balanced_accuracy` de val com `patience=3`.

Tempo esperado em T4: **~1,5-3h** dependendo de quantas épocas o early stopping deixar correr.

**Importante:** os outputs salvam direto no Drive pelo symlink. Mesmo se a sessão for revogada, history.json, checkpoint_best.pt e final_test_metrics.json continuam lá.

In [ ]:
!python -m src.training.run \
    --epochs 20 \
    --batch-size 32 --batch-size-eval 64 \
    --lr 1e-4 --weight-decay 0.05 \
    --num-workers 2 \
    --run-name resnet50_class3_weighted

## 9. Análise dos resultados

Carrega `history.json` e `final_test_metrics.json` da run mais recente e plota:
- Curvas de loss (train vs val)
- Curvas de balanced accuracy e macro-F1 por época
- AUC macro por época
- Matriz de confusão final no test set

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Encontra a run mais recente do tipo 'resnet50_class3_weighted'
runs_dir = Path('experiments/runs')
runs = sorted([d for d in runs_dir.iterdir() if d.is_dir() and 'weighted' in d.name and 'smoke' not in d.name])
assert runs, 'Nenhuma run de treino real encontrada.'
run = runs[-1]
print(f'Analisando run: {run.name}')

history = json.loads((run / 'history.json').read_text(encoding='utf-8'))
final = json.loads((run / 'final_test_metrics.json').read_text(encoding='utf-8'))

epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history]
bal_acc = [h['val_metrics']['balanced_accuracy'] for h in history]
macro_f1 = [h['val_metrics']['macro_f1'] for h in history]
auc = [h['val_metrics']['auc_macro'] or 0 for h in history]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(epochs, train_loss, 'o-', label='train')
axes[0, 0].plot(epochs, val_loss, 's-', label='val')
axes[0, 0].set_title('Loss por epoca'); axes[0, 0].legend(); axes[0, 0].set_xlabel('epoca')

axes[0, 1].plot(epochs, bal_acc, 'o-', color='C2', label='balanced acc')
axes[0, 1].plot(epochs, macro_f1, 's-', color='C3', label='macro F1')
axes[0, 1].set_title('Metricas balanceadas (val)'); axes[0, 1].legend()
axes[0, 1].set_xlabel('epoca'); axes[0, 1].set_ylim(0, 1)

axes[1, 0].plot(epochs, auc, 'o-', color='C4')
axes[1, 0].set_title('AUC macro (val)'); axes[1, 0].set_xlabel('epoca'); axes[1, 0].set_ylim(0.5, 1)

cm = np.array(final['test_metrics']['confusion'])
class_names = final['test_metrics']['class_names']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1, 1])
axes[1, 1].set_title('Matriz de confusao (test)'); axes[1, 1].set_xlabel('predito'); axes[1, 1].set_ylabel('real')

plt.tight_layout(); plt.savefig(run / 'final_curves.png', dpi=130); plt.show()

print('\n=== Metricas finais no TEST ===')
tm = final['test_metrics']
for k in ['accuracy', 'balanced_accuracy', 'macro_f1', 'auc_macro']:
    print(f'  {k:20s}: {tm[k]:.4f}')
print('  Por classe:')
for cls in class_names:
    p = tm['precision_per_class'][cls]
    r = tm['recall_per_class'][cls]
    f = tm['f1_per_class'][cls]
    print(f'    {cls:20s} P={p:.3f}  R={r:.3f}  F1={f:.3f}')

## 10. Ablação obrigatória — treino sem pesos de classe

ADR-0007 estabelece que **comparar com vs sem peso** é obrigatório para sustentar empiricamente a escolha do weighted CE. Se a `balanced_accuracy` cair significativamente sem peso, validamos a decisão.

Mesmo seed (`42`), mesmo split, mesmas outras condições — só muda a loss.

In [ ]:
!python -m src.training.run \
    --epochs 20 \
    --batch-size 32 --batch-size-eval 64 \
    --lr 1e-4 --weight-decay 0.05 \
    --num-workers 2 \
    --no-class-weights \
    --run-name resnet50_class3_noweight

In [ ]:
# Comparativo lado a lado: com peso vs sem peso
import pandas as pd

rows = []
for name_filter in ['weighted', 'noweight']:
    matching = sorted([
        d for d in runs_dir.iterdir()
        if d.is_dir() and name_filter in d.name and 'smoke' not in d.name
    ])
    if not matching:
        print(f'(sem run "{name_filter}" ainda)')
        continue
    r = matching[-1]
    f = json.loads((r / 'final_test_metrics.json').read_text(encoding='utf-8'))
    tm = f['test_metrics']
    rows.append({
        'run': r.name,
        'accuracy': tm['accuracy'],
        'balanced_accuracy': tm['balanced_accuracy'],
        'macro_f1': tm['macro_f1'],
        'auc_macro': tm['auc_macro'],
        'F1_non_demented': tm['f1_per_class'].get('non_demented'),
        'F1_very_mild': tm['f1_per_class'].get('very_mild'),
        'F1_mild_or_moderate': tm['f1_per_class'].get('mild_or_moderate'),
    })
pd.DataFrame(rows).set_index('run').T

## 11. Próximos passos

1. **Anote os números** das duas runs no documento de experimentos do TCC (formato sugerido no CLAUDE.md: planilha simples).
2. **Confirme com o orientador** os resultados antes de avançar para os modelos transformer.
3. **Implementar ViT-Base/16** (mesma estrutura: `src/models/vit.py` + `build_vit_base16(num_classes, pretrained=True)`), depois Swin-T.
4. **Comparar via McNemar** (`src/evaluation/metrics.py::mcnemar_test`) entre ResNet, ViT, Swin no mesmo test set.
5. **Fase de interpretabilidade** (semana 4): Attention Rollout + Grad-CAM sobre o melhor modelo.

Todos os outputs deste notebook ficam preservados em `MyDrive/TCC/runs/` mesmo após o término da sessão Colab.